In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
将 NEU-DET 数据集的 VOC 格式 XML 标注转换为 YOLO 格式 TXT。
适配目录结构：
    NEU-DET/
        ANNOTATIONS/   (存放 *.xml)
        JPEGImages/    (存放 *.jpg)
输出：在 NEU-DET 下创建 labels/ 目录，存放 *.txt
"""

import os
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm
from PIL import Image

# NEU-DET 的 6 个类别（顺序固定，与训练时的 data.yaml 一致）
CLASSES = [
    "crazing",
    "inclusion",
    "patches",
    "pitted_surface",
    "rolled-in_scale",
    "scratches"
]

def convert_xml_to_yolo(xml_path, img_path, output_txt_path):
    """转换单个 XML 文件为 YOLO TXT"""
    # 读取图像尺寸
    with Image.open(img_path) as img:
        width, height = img.size

    tree = ET.parse(xml_path)
    root = tree.getroot()

    with open(output_txt_path, 'w') as f:
        for obj in root.findall('object'):
            class_name = obj.find('name').text
            if class_name not in CLASSES:
                print(f"警告: 未知类别 '{class_name}' in {xml_path.name}")
                continue
            class_id = CLASSES.index(class_name)

            bndbox = obj.find('bndbox')
            xmin = float(bndbox.find('xmin').text)
            ymin = float(bndbox.find('ymin').text)
            xmax = float(bndbox.find('xmax').text)
            ymax = float(bndbox.find('ymax').text)

            # 归一化
            x_center = (xmin + xmax) / 2.0 / width
            y_center = (ymin + ymax) / 2.0 / height
            box_width = (xmax - xmin) / width
            box_height = (ymax - ymin) / height

            # 裁剪到 [0,1] 防止边界溢出
            x_center = max(0.0, min(1.0, x_center))
            y_center = max(0.0, min(1.0, y_center))
            box_width = max(0.0, min(1.0, box_width))
            box_height = max(0.0, min(1.0, box_height))

            f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}\n")

def main():
    # 指定你的 NEU-DET 根目录（脚本会自动检测）
    neu_root = Path("/root/rivermind-data/gc/NEU-DET")
    xml_dir = neu_root / "ANNOTATIONS"
    img_dir = neu_root / "JPEGImages"
    output_dir = neu_root / "labels"   # 新建 labels 目录存放 YOLO txt
    output_dir.mkdir(exist_ok=True)

    xml_files = list(xml_dir.glob("*.xml"))
    if not xml_files:
        print(f"错误: {xml_dir} 中没有 XML 文件")
        return

    print(f"找到 {len(xml_files)} 个 XML 文件，开始转换...")
    for xml_path in tqdm(xml_files, desc="转换进度"):
        # 对应的图片文件（假设扩展名为 .jpg）
        img_name = xml_path.stem + ".jpg"
        img_path = img_dir / img_name
        if not img_path.exists():
            print(f"警告: 图片不存在 {img_path}，跳过 {xml_path.name}")
            continue

        output_txt = output_dir / (xml_path.stem + ".txt")
        convert_xml_to_yolo(xml_path, img_path, output_txt)

    print(f"转换完成！YOLO 标签保存在: {output_dir}")
    print("接下来你可以将 images 和 labels 按 train/val 划分，然后修改训练脚本的 num_classes=6。")

if __name__ == "__main__":
    main()

找到 1800 个 XML 文件，开始转换...


转换进度: 100%|██████████| 1800/1800 [00:00<00:00, 3035.59it/s]

转换完成！YOLO 标签保存在: /root/rivermind-data/gc/NEU-DET/labels
接下来你可以将 images 和 labels 按 train/val 划分，然后修改训练脚本的 num_classes=6。
